# w5_cpu_o2_joint_v2_5seed


In [ ]:
import hashlib, io, json, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
MOUNT = Path('/kaggle/input')
assert WORK.is_dir(), 'Run on Kaggle'
REV = 'e1696028c3fe589a96797f7afee04f03d9129f73'
with tempfile.TemporaryDirectory(prefix='w4_ja_') as ts:
    tmp = Path(ts)
    repo = tmp / 'repo'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'CUDA_VISIBLE_DEVICES': ''}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    dest = WORK / 'pinns_o2_joint_v2_5seed.zip'
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_pinns_o2_joint_v2_5seed', str(joint_bin), str(pose_npz), str(dest)], cwd=repo, env=env, check=True)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
